# Avaliação e análise dos resultados

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Igorbraziel/juristcu-clustering-tsdae/blob/main/notebooks/04_evaluation.ipynb)

- **Objetivo:** medir NMI, ARI e silhueta e testar H1 e H2.
- **Entradas:** tabelas do `03_clustering` em `results/tables/`; áreas, temas e enunciados do JurisTCU.
- **Saídas:** tabelas em `results/tables/` e figuras em `results/figures/`.
- **Requisitos:** não requer GPU nem o Google Drive. As métricas com áreas e temas são calculadas apenas neste notebook.

In [ ]:
# Configuração do ambiente. Fora do Colab, usa o ambiente local criado por `uv sync`.
# O Google Drive não é montado: o notebook lê apenas tabelas versionadas em results/.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # Trocar pelo nome de uma branch de trabalho para testá-la.
REPO_URL = "https://github.com/Igorbraziel/juristcu-clustering-tsdae.git"
REPO_DIR = "/content/juristcu-clustering-tsdae"


def run(command: str) -> None:
    """Executa um comando de shell e interrompe a célula em caso de erro."""
    subprocess.run(command, shell=True, check=True)


if IN_COLAB:
    # Clona o repositório, se necessário, e posiciona na versão remota da branch.
    run(f"test -d {REPO_DIR} || git clone -q {REPO_URL} {REPO_DIR}")
    run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
    run(f"git -C {REPO_DIR} checkout -q -B {BRANCH} FETCH_HEAD")
    # O uv costuma vir no Colab. Instala apenas se faltar.
    run("command -v uv > /dev/null || pip install -q uv")
    # --system: instala no Python do Colab, que já traz o torch com CUDA.
    # --no-sources: ignora o índice de torch CPU do pyproject.toml.
    # --no-config: ignora configurações locais do uv.
    # Instalação não editável: disponível sem reiniciar o ambiente de execução.
    # Nunca usar `uv sync` nem requirements.txt no Colab.
    run(f"uv pip install --system --no-config --no-sources -q {REPO_DIR}")
    # Tabelas e figuras são lidas e gravadas em results/, a partir da raiz do repositório.
    os.chdir(REPO_DIR)
    # Após atualizar o repositório com mudanças em src/, reiniciar o ambiente de execução.

In [ ]:
import importlib

import numpy as np
import pandas as pd

# Atualiza o cache de importação para reconhecer o pacote recém-instalado.
importlib.invalidate_caches()

from juristcu import analysis, data, evaluation, storage  # noqa: E402
from juristcu.config import (  # noqa: E402
    AREA_COLUMN,
    MIN_TOPIC_SIZE,
    N_BOOTSTRAP,
    REPRESENTATION_NAMES,
    SEED,
    TEXT_COLUMN,
    TOPIC_COLUMN,
    TSDAE_SEEDS,
)

# Apenas exibe o ambiente. save_environment gravaria no diretório de artefatos.
storage.log_environment()

## 1. Carregamento

- Documentos do JurisTCU, na ordem de `data.load_documents`.
- Enunciados limpos por `data.clean_text`, como nas representações.
- Agrupamentos e silhuetas gravados pelo `03_clustering` em `results/tables/`.
- A conferência de alinhamento interrompe o notebook se as tabelas não corresponderem aos documentos.

In [ ]:
documents = data.load_documents(columns=(TEXT_COLUMN, AREA_COLUMN, TOPIC_COLUMN))
# Limpeza do HTML, igual à das representações.
texts = documents[TEXT_COLUMN].map(data.clean_text).tolist()
areas = documents[AREA_COLUMN].to_numpy()
topics = documents[TOPIC_COLUMN].to_numpy()
n_areas = documents[AREA_COLUMN].nunique()

# Temas com ao menos MIN_TOPIC_SIZE documentos, o mesmo filtro do 03_clustering.
topic_mask = evaluation.filter_min_size(topics, min_size=MIN_TOPIC_SIZE)
kept_topics = topics[topic_mask]
n_topics = documents.loc[topic_mask, TOPIC_COLUMN].nunique()

r3_names = [f"R3-seed{seed}" for seed in TSDAE_SEEDS]
n_areas, n_topics

In [ ]:
area_assignments = storage.load_table("area_assignments")
topic_assignments = storage.load_table("topic_assignments")
cluster_silhouette = storage.load_table("cluster_silhouette")

# Conferência de alinhamento: cada linha corresponde a um documento de data.load_documents.
expected_columns = ["document", *REPRESENTATION_NAMES]
for table in (area_assignments, topic_assignments):
    assert list(table.columns) == expected_columns, f"Colunas inesperadas: {list(table.columns)}."
assert len(area_assignments) == len(documents), "area_assignments não cobre todos os documentos."
assert np.array_equal(area_assignments["document"], np.arange(len(documents))), (
    "area_assignments fora da ordem dos documentos."
)
assert np.array_equal(topic_assignments["document"], np.flatnonzero(topic_mask)), (
    "topic_assignments não corresponde aos temas mantidos."
)
assert set(r3_names) <= set(REPRESENTATION_NAMES), "Sementes de R3 ausentes das tabelas."

area_preds = {name: area_assignments[name].to_numpy() for name in REPRESENTATION_NAMES}
topic_preds = {name: topic_assignments[name].to_numpy() for name in REPRESENTATION_NAMES}
len(area_assignments), len(topic_assignments)

## 2. Métricas por área

- NMI (principal) e ARI em relação às áreas.
- IC de 95% da NMI por bootstrap dos documentos, sobre as atribuições já obtidas.
- Silhueta calculada pelo `03_clustering`, que dispõe das representações.
- R3-mean: média das 3 sementes, com IC da NMI média.

In [ ]:
area_silhouette = cluster_silhouette[cluster_silhouette["analysis"] == "area"].set_index(
    "representation", verify_integrity=True
)
# A silhueta precisa vir do mesmo k-means das atribuições.
assert (area_silhouette["k"] == n_areas).all(), "Silhueta por área com k diferente do esperado."

area_cis = {
    name: evaluation.bootstrap_nmi_ci(areas, [preds], n_resamples=N_BOOTSTRAP, seed=SEED)
    for name, preds in area_preds.items()
}
area_scores = pd.DataFrame(
    {
        name: {
            **evaluation.external_scores(areas, preds),
            "nmi_low": area_cis[name].low,
            "nmi_high": area_cis[name].high,
            "silhouette": area_silhouette.loc[name, "silhouette"],
        }
        for name, preds in area_preds.items()
    }
).T
# R3 pela média das 3 sementes, com IC da NMI média.
r3_area_ci = evaluation.bootstrap_nmi_ci(
    areas, [area_preds[name] for name in r3_names], n_resamples=N_BOOTSTRAP, seed=SEED
)
area_scores.loc["R3-mean"] = area_scores.loc[r3_names].mean()
area_scores.loc["R3-mean", ["nmi_low", "nmi_high"]] = [r3_area_ci.low, r3_area_ci.high]
storage.save_table(area_scores.rename_axis("representation").reset_index(), "area_scores")
area_scores

## 3. Testes de H1 e H2

- Diferença de NMI com IC de 95% por bootstrap pareado dos documentos. R3 entra pela média das sementes.
- Critério: limite inferior do IC positivo.
- H1 se confirma quando as duas comparações de H1 atendem ao critério. H2 depende só de R3 − R2.
- R3 − R2-CLS é exploratória e não altera a decisão sobre H2. Ver `docs/decisions.md`.

In [ ]:
r3_area_preds = [area_preds[name] for name in r3_names]
comparisons = {
    "R3 - R1": ("H1", r3_area_preds, [area_preds["R1"]]),
    "R4 - R1": ("H1", [area_preds["R4"]], [area_preds["R1"]]),
    "R3 - R2": ("H2", r3_area_preds, [area_preds["R2"]]),
    "R3 - R2-CLS": ("exploratória", r3_area_preds, [area_preds["R2-CLS"]]),
}
test_rows = []
for label, (hypothesis, preds_a, preds_b) in comparisons.items():
    ci = evaluation.bootstrap_nmi_difference(
        areas, preds_a, preds_b, n_resamples=N_BOOTSTRAP, seed=SEED
    )
    test_rows.append(
        {
            "comparison": label,
            "hypothesis": hypothesis,
            "difference": ci.estimate,
            "low": ci.low,
            "high": ci.high,
            "positive_lower_bound": bool(ci.low > 0),
        }
    )
hypothesis_tests = pd.DataFrame(test_rows)
storage.save_table(hypothesis_tests, "hypothesis_tests")
hypothesis_tests

In [ ]:
# Uma hipótese se confirma quando todas as suas comparações têm limite inferior positivo.
hypothesis_decisions = (
    hypothesis_tests[hypothesis_tests["hypothesis"].isin(["H1", "H2"])]
    .groupby("hypothesis", as_index=False)
    .agg(comparisons=("comparison", "; ".join), confirmed=("positive_lower_bound", "all"))
)
storage.save_table(hypothesis_decisions, "hypothesis_decisions")
hypothesis_decisions

## 4. Métricas por tema

- Apenas os documentos dos temas com ao menos 20 documentos, filtrados antes do k-means.
- k igual ao número de temas mantidos.
- Mesmas métricas e mesmo bootstrap da análise por área, inclusive a linha R3-mean.

In [ ]:
topic_silhouette = cluster_silhouette[cluster_silhouette["analysis"] == "topic"].set_index(
    "representation", verify_integrity=True
)
assert (topic_silhouette["k"] == n_topics).all(), "Silhueta por tema com k diferente do esperado."

topic_cis = {
    name: evaluation.bootstrap_nmi_ci(kept_topics, [preds], n_resamples=N_BOOTSTRAP, seed=SEED)
    for name, preds in topic_preds.items()
}
topic_scores = pd.DataFrame(
    {
        name: {
            **evaluation.external_scores(kept_topics, preds),
            "nmi_low": topic_cis[name].low,
            "nmi_high": topic_cis[name].high,
            "silhouette": topic_silhouette.loc[name, "silhouette"],
        }
        for name, preds in topic_preds.items()
    }
).T
# R3 pela média das 3 sementes, com IC da NMI média.
r3_topic_ci = evaluation.bootstrap_nmi_ci(
    kept_topics, [topic_preds[name] for name in r3_names], n_resamples=N_BOOTSTRAP, seed=SEED
)
topic_scores.loc["R3-mean"] = topic_scores.loc[r3_names].mean()
topic_scores.loc["R3-mean", ["nmi_low", "nmi_high"]] = [r3_topic_ci.low, r3_topic_ci.high]
storage.save_table(topic_scores.rename_axis("representation").reset_index(), "topic_scores")
topic_scores

## 5. Número de grupos pela silhueta (P3)

- Silhueta do k-means para k de 5 a 50, calculada pelo `03_clustering` sem consultar as áreas.
- Figura 2: silhueta em função de k, por representação.
- O k de maior silhueta é comparado ao número de áreas.

In [ ]:
silhouette_by_k = storage.load_table("silhouette_by_k")
best_k = storage.load_table("best_k")

# Uma curva por representação, no formato de clustering.silhouette_sweep.
sweeps = {
    name: silhouette_by_k.loc[
        silhouette_by_k["representation"] == name, ["k", "silhouette"]
    ].reset_index(drop=True)
    for name in REPRESENTATION_NAMES
}
silhouette_figure = analysis.plot_silhouette_curves(sweeps)
storage.save_figure(silhouette_figure, "silhouette_by_k")

In [ ]:
best_k

## 6. Projeções UMAP

- Figura 1: projeção em 2 dimensões de cada representação, colorida por área.
- Projeções calculadas pelo `03_clustering`, na ordem dos documentos.
- O pre-commit rejeita arquivos acima de 1 MB. Se a figura passar desse limite, reduzir o `dpi` de `storage.save_figure`.

In [ ]:
projections = {
    name: storage.load_table(f"umap_{name}")[["x", "y"]].to_numpy() for name in REPRESENTATION_NAMES
}
assert all(len(xy) == len(documents) for xy in projections.values()), (
    "Projeção UMAP com número de linhas diferente do número de documentos."
)
umap_figure = analysis.plot_umap_grid(projections, colors=areas)
storage.save_figure(umap_figure, "umap_by_area")

## 7. Termos característicos

- Termos TF-IDF de cada grupo de R3 (primeira semente), a partir dos enunciados limpos.
- Composição de cada grupo por área, para comparar os termos com as áreas.

In [ ]:
reference_preds = area_preds[r3_names[0]]
top_terms = analysis.top_terms_per_cluster(texts, reference_preds, n_terms=10)
top_terms_table = pd.DataFrame(
    {"group": list(top_terms), "terms": [", ".join(terms) for terms in top_terms.values()]}
)
storage.save_table(top_terms_table, "top_terms")
top_terms_table

In [ ]:
group_area_crosstab = pd.crosstab(reference_preds, areas, rownames=["group"], colnames=["area"])
storage.save_table(group_area_crosstab.reset_index(), "group_area_crosstab")
group_area_crosstab

## 8. HDBSCAN (opcional)

- Análise exploratória. Não é comparável ao k-means: o número de grupos não é fixado e parte dos documentos fica como ruído.
- NMI e ARI em relação às áreas, apenas nos documentos fora do ruído.
- Número de grupos e fração de ruído calculados pelo `03_clustering`.
- Primeiro item da ordem de corte. Se as tabelas do HDBSCAN faltarem, `storage.load_table` levanta `FileNotFoundError` e esta seção pode ser omitida.

In [ ]:
hdbscan_assignments = storage.load_table("hdbscan_assignments")
hdbscan_summary = storage.load_table("hdbscan_summary")
assert list(hdbscan_assignments.columns) == expected_columns, (
    "Colunas inesperadas em hdbscan_assignments."
)
assert np.array_equal(hdbscan_assignments["document"], np.arange(len(documents))), (
    "hdbscan_assignments fora da ordem dos documentos."
)

hdbscan_rows = []
for name in REPRESENTATION_NAMES:
    preds = hdbscan_assignments[name].to_numpy()
    # O rótulo -1 indica ruído e fica fora da NMI e da ARI.
    mask = preds != -1
    hdbscan_rows.append(
        {"representation": name, **evaluation.external_scores(areas[mask], preds[mask])}
    )
hdbscan_scores = pd.DataFrame(hdbscan_rows).merge(
    hdbscan_summary, on="representation", validate="one_to_one"
)
storage.save_table(hdbscan_scores, "hdbscan_scores")
hdbscan_scores